In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import krippendorff
import pingouin as pg
from typing import Tuple, Literal


In [2]:
## Load Data

human_turn = pd.read_csv('human_turn.csv')
human_holistic = pd.read_csv('human_holistic.csv')
llm_turn = pd.read_csv('llm_turn.csv')
llm_holistic = pd.read_csv('llm_holistic.csv')

In [3]:
human_turn.head()

,rater_id,group_id,scenario_id,turn,metric_id,number_id,score
0,rater_01,Group A,L1_frustrated_multi,1,tone_appropriateness,1,4
1,rater_01,Group A,L5_neutral_multi,1,tone_appropriateness,1,4
2,rater_01,Group A,L10_angry_multi,1,tone_appropriateness,1,4
3,rater_02,Group B,L10_neutral_multi,1,tone_appropriateness,1,3
4,rater_03,Group B,L10_neutral_multi,1,tone_appropriateness,1,4


In [4]:
print(human_turn.group_id.value_counts())
print(human_turn.metric_id.value_counts())
print(human_turn.number_id.value_counts())

group_id
Group A    225
Group B    225
Group C    225
Name: count, dtype: int64
metric_id
tone_appropriateness          135
emotional_calibration         135
emotional_escalation          135
functional_empathy            135
contextual_appropriateness    135
Name: count, dtype: int64
number_id
1    135
2    135
3    135
4    135
5    135
Name: count, dtype: int64


## Restructure Data for Krippendorff's Alpha

In [5]:
def get_all_reliability_matrices(df, group, rating_type='turn'):
    """
    Returns ALL reliability matrices for a group in one dictionary.
    
    Parameters
    ----------
    df : DataFrame
        Human ratings (human_turn.csv or human_holistic.csv)
    group : str
        'Group A', 'Group B', or 'Group C'
    rating_type : str
        'turn' (turns 1-5) or 'holistic' (turn == 0)
    
    Returns
    -------
    dict
        Keys = formatted names like 'A_tone_appropriateness'
        Values = wide-format matrices (items × raters)
    """
    # Filter to target group
    group_data = df[df['group_id'] == group].copy()
    if group_data.empty:
        raise ValueError(f"No data for {group}")
    
    # Get all unique metrics in this group's data
    metrics = sorted(group_data['metric_id'].unique())
    
    # Extract group letter (e.g., 'A' from 'Group A')
    group_letter = group.split()[-1]
    
    # Build dictionary of matrices
    matrices = {}
    for metric in metrics:
        # Filter to this metric + rating type
        subset = group_data[group_data['metric_id'] == metric].copy()
        
        if rating_type == 'turn':
            subset = subset[subset['turn'].between(1, 5)]
        else:  # holistic
            subset = subset[subset['turn'] == 0]
        
        # Pivot to wide format: (scenario, turn) × raters
        wide = subset.pivot(
            index=['scenario_id', 'turn'],
            columns='rater_id',
            values='score'
        )
        
        # Store with clean key name
        key = f"{group_letter}_{metric}"
        matrices[key] = wide
    
    print(f"✓ Created {len(matrices)} matrices for {group} ({rating_type}-level)")
    print(f"  Keys: {list(matrices.keys())}")
    return matrices

In [6]:
turn_A = get_all_reliability_matrices(human_turn, 'Group A', rating_type='turn')
turn_B = get_all_reliability_matrices(human_turn, 'Group B', rating_type='turn')
turn_C = get_all_reliability_matrices(human_turn, 'Group C', rating_type='turn')

✓ Created 5 matrices for Group A (turn-level)
  Keys: ['A_contextual_appropriateness', 'A_emotional_calibration', 'A_emotional_escalation', 'A_functional_empathy', 'A_tone_appropriateness']
✓ Created 5 matrices for Group B (turn-level)
  Keys: ['B_contextual_appropriateness', 'B_emotional_calibration', 'B_emotional_escalation', 'B_functional_empathy', 'B_tone_appropriateness']
✓ Created 5 matrices for Group C (turn-level)
  Keys: ['C_contextual_appropriateness', 'C_emotional_calibration', 'C_emotional_escalation', 'C_functional_empathy', 'C_tone_appropriateness']


In [7]:
turn_B['B_tone_appropriateness'].head()

rater_id                rater_02  rater_03  rater_09
scenario_id       turn                              
L10_neutral_multi 1            3         4         4
                  2            3         4         3
                  3            3         4         2
                  4            3         5         3
                  5            2         5         4

In [14]:
def compute_krippendorff_alpha(matrix):

    # Convert to numpy array for krippendorff
    data = matrix.values.astype(float)
    
    # Compute alpha with ordinal measurement level
    alpha = krippendorff.alpha(
        reliability_data=data,  # krippendorff expects raters × items
        level_of_measurement='interval'
    )
    return alpha

In [15]:
def compute_reliability_table(turn_A, turn_B, turn_C):
    """
    Compute Krippendorff's Alpha (ordinal) for all matrices and organize into table.
    
    Parameters
    ----------
    turn_A, turn_B, turn_C : dict
        Dictionaries of matrices from get_all_reliability_matrices()
        Keys like 'A_contextual_appropriateness', values are wide-format DataFrames
    
    Returns
    -------
    pd.DataFrame
        Table with metrics as index and groups as columns
    """
    # Map group labels to their matrix dictionaries
    group_data = {
        'Group A': turn_A,
        'Group B': turn_B,
        'Group C': turn_C
    }
    
    # Extract all unique metrics across groups (strip group prefix)
    all_metrics = set()
    for matrices in group_data.values():
        for key in matrices.keys():
            metric = '_'.join(key.split('_')[1:])  # Remove 'A_', 'B_', etc.
            all_metrics.add(metric)
    all_metrics = sorted(all_metrics)
    
    # Build results table
    results = []
    for metric in all_metrics:
        row = {'metric': metric}
        for group_name, matrices in group_data.items():
            # Reconstruct key (e.g., 'A_contextual_appropriateness')
            group_letter = group_name.split()[-1]
            key = f"{group_letter}_{metric}"
            
            if key in matrices:
                alpha = compute_krippendorff_alpha(matrices[key])
                row[group_name] = alpha
            else:
                row[group_name] = np.nan
        results.append(row)
    
    # Create and format DataFrame
    df = pd.DataFrame(results).set_index('metric')
    df.index.name = 'metric'
    df = df[['Group A', 'Group B', 'Group C']].round(3)
    
    print(f"✓ Computed Krippendorff's Alpha (ordinal) for {len(all_metrics)} metrics across 3 groups")
    return df

In [16]:
reliability_table = compute_reliability_table(turn_A, turn_B, turn_C)
print(reliability_table)

✓ Computed Krippendorff's Alpha (ordinal) for 5 metrics across 3 groups
                            Group A  Group B  Group C
metric                                               
contextual_appropriateness   -0.045    0.455    0.161
emotional_calibration        -0.030    0.474    0.092
emotional_escalation         -0.020    0.118    0.185
functional_empathy            0.078    0.368    0.121
tone_appropriateness          0.016    0.399    0.214
